# Đề 12 – Kaggle GPU (T4): DINOv3 + SAM 3 region/re-rank trên benchmark 70 video
Code repo được nhúng (tar.gz base64) trong cell đầu – không cần GH_TOKEN. Cần gắn secret `HF_TOKEN` (Add-ons → Secrets); thiếu thì notebook dừng ngay.

In [ ]:
import os, sys, base64, io, tarfile, time
# Secret HF_TOKEN must be attached to this notebook (Add-ons -> Secrets). Never printed.
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['HF_TOKEN'] = UserSecretsClient().get_secret('HF_TOKEN')
    print('HF_TOKEN secret: attached')
except Exception as e:
    print('HF_TOKEN secret NOT attached:', type(e).__name__); os.makedirs('/kaggle/working/results', exist_ok=True)
    open('/kaggle/working/results/STATUS.txt','w').write('HF_TOKEN secret not attached - enable it in Add-ons > Secrets and rerun\n'); raise SystemExit('stop: no HF_TOKEN')
os.makedirs('/kaggle/working/de12', exist_ok=True); os.chdir('/kaggle/working/de12')
CODE = 'H4sIALN522oC/+19XXPbSJJgP/NX1KB7pgGJhAh+ytTQ0f5Qu31tt2TJ7Z4JmosBSZCEBQIQAFLiyLrYjY24iHu7vX2+l9vYe7i4h3u7iOmHe9iN+x+eX3KZWVX44Ifl7nGrZ1qAwyI+6jMrKyszKzNL39P3vji2Lr+yrZEdfvKTXFV+bfutVuuN9B7fG9WaUfuEXX5yC9c8iq0Qqv/kbl61fTaLnZndNdr3au1Ws3GvrberLaNVb5Q+Ka5f/LWIbCscTvd+yjpwUrebTfw12s1q9jeZ80az1oKrXW3XcP63gSSw5m3O/4H/fnJz0/e/0Uv/q6D/9XX6bxT0/1bofztL/5v7LaOpN9uN/bZRkP+7RP9N0/Gc2DT1YPmTzP9Wo/EB9L9pNOs4/+vVWv0TVr3N+V/Q/4L/L/h/o9Fo6vVWbb9RbxYLwB2i/0M/tH8K2n8z/QdWz2hL+t82GiAnGLVao1rQ/9u4FEV55Yxsnzkza2Lv+YM39jBmHCk6bBxaMztilftMHc9dt0LPbJeF9sTxPY3Zs4E9GjnehNJ8+eDp6SneRPZkZntxpEPpJWcW+GHM3kS+V2Z+VGYT1x+UGRId+W24qJWZN58FS2ZFzAvKbGw5ESSNfWhGaRz6M3b89BkTyZ9iU/nbOLS8aOyHMzuM5OcH89h/7o9st0y3x6E/tKPID/kjZU7elUqvnj4+PDIPf/eSdZmq6LOgoZSZolsLh35n/oL/nvHfC3swU7RS6VNWWbkSYK1cpZE9ZtbUiqbqONQ6JQbXBCqDTuuhHTl/tFW8HS7iR77rh5CoTN8eHT07OjEfPjmpPTl58HutzNT9MtuHX8eL7TDwXSuGMehi0qffvDw8MR+cHD7QqPjQjuehx9QJu88m+sy2PFXT9DHkiG24LVGTImsWuLbJW60GVjwFsAdR19CrZTayR/OgWxXNhWH81nMQzDwXDLjOk9yvdlh05gTM4t1nF1M/slmrURk4MbO+gl4zJ2IXTjx1PPYHyvMHBt8i5o9ZPLWZa0UxO7ODmBdA9amAkRYb+GHoX9gjRiP92Jl8G2g6+71ju6OIqbEZ2cMyooVOQ6oRsmFuKrHLvvE9XtrQCgS4CdMfWQFAx6YeawdsAX3Gz1agT+yYhuLRg2Pz+OTo2Pzy+FRjfshqTb1KRUWxDcjpQPqZdakaZRb6c2+kUhl7CD0NxocnvZg6rs1ehnObwxAv/wxAHIraQuC4VC355oyZ58eQpMMG8Oks+8Fhv6aqWbfLqmlxeE2htAS5MC2BmNkujEICgUxR/LPljTiYYGywVkxJLwG3VHXKftWlz5oezWeAOuy3XZ4xXzcVyXa7zDhgQ9+LHW+er0+MxDT3cokDCPn2CPRlPpt1HGMrDK3lTZPh5MlDLQM1ql4OM8DUta3IVjdPUE61tszQSeiMTJFCdWaTMnPtBYCxqwJtqsOw+gs7dC2YFHqtmc6Lx3YMBNMPgTTatqiBBaEf+JHlRh2ZK4Apw2ZzN3Yq0dACxMDqdHZCEzVirgOAggmhXlbLy2r50igvjRShvyuzrwCM0CgdycUBzIxLG5G216fvMDGZB0MnWpwO0vACoIcY8h1A28NS4OeABfA6wNfDC7YjW0gpk6ekCCw7xLKB0E5s1dPyKICfh9s/40WN1QEENswUFedNlWiYOtwZXlSCC5wymbfhznBaCab01vHU7/hbdbhraJB+V6SHL1+JLyF9me5iHi1H/6jmUmkIaBixo++eLWonfHyT0TsKkIZaLvOhdZWFP7QGYgn0YHlIx5GpT46/hWKH/gwWNpgKQIgmvj9xYcW8cBe1ygCwrgIUZTg1WhXbi2BhdO1kABG/pICtRrY7htUOaF1XubkMBRfB4Ky7jzR54Qwh0zCYKxk4b10Hj7DUzOpHz1/64RF1kCMu9D4pCBumI6nMZ6SpaQYA09ByPHukYtOBcFLymUy+Xu7mfHrsq7wjmm4vLDdDAalAoJD0C2mgaOy67Dil+4I4At3zzUmYkE8OXphWbgJemCsZEF1CUbx3KrE5UZcmOEcTE9ZE6GbUVYJYofbJBqRNG89oDeFd1qkMk/M+dqgGzqXtmtCXORR82VOyz0pf61X7STnA98AqC9NvBKVhoXo0tQKY0WPbiiP5DrgCfKti6h1MnDYEkDNtR4qpCOaRg3RIpZKAcDuTme8AhHLVC7oh8sPjasYytQAzMfYpG14OlxdTnN6BBV0dVaLzuRXaQOz9ENbgXrVspIWfilVRkilAkTBLpSS5eIPlQdN1HFwO6zNNdzxoBtCJ2EdaqK7QkeElkKdlmQ0AGgOkXCp1pfemDyTrVEty5TIBLWVATBlQU7Y0ROvgeXhZGVzs1VKyM1xWBlP+AuiKbD/AANPuyrTZTwZ+Wu5SrtUV9tKoXFaB8TJatKIujcqSP3YQHgkhRNp1WeXsnLqUN5eGfGNomWVOELQQydmGhU2i4vrKxmnfqYOs3mPHS5nhhAS+xAm1cCKYsXwNs8/nDiAv8O+4Ij1++s3Rov7KeZnnm3HOWbCcRdQpAAkDke0S/gNvB+whiA9iYf/GjhkyoHtRPHofPcQyupA/M/CEGlR2l/F1j+MtlAZvvEDn/EKvqjf2m8B26Y1mi36qrb6kT1DrStpa7R4mgqr4T7P/HiKCtGIzncgTF6gAxMjhmdpTsa6I1+bMdGCLYEGNVQXYFiAugt9Xk66V015qkhV6+PTZ028OH5wA37UHrCfwnqySdh1fJl3D+eTMcELxtvbXUOYqT446XKTSeV/US50WDljkbJwI9TKDuYLYPoqXgd3laceub8X1mnYtl9JDgW4JDg1dJ+jQGmo5e/hQWThxZj2r19hbGMIJfKl15MIpnvPLHmLQWwYSpb+AlGNraA98/2yPv6hEMxgeLiHQmzqzdZBE8snqWHkEZcmlp+IuRkZr/94Mud2JFdujDvNsGwjYV18COM5sDyRaF6gPLLnMGg5BFrG8YUag2Lp8v6/HSrJgI3O9itaYH/EGfg6yCx7PgjKHqgznI0tBisKHAR91JzKtheW4FrAGqsbZfM4Q5MvHdTmRg7evxMlKt3ktBqIM5SgIVmqIvPOo4aL2hatkZZVMTpk1T8vjcNlJeY11kXxza3NF2Jc4SDBdkM85BCkxzJS4gdgBpGA146QMmTpBwzgSxCkBhGUlJHzwiXlMK8WerjT6Q9r74dwKJzSDqAvzLCMvzuP19dNJ2e0qCime4Gk0zL+ybm5ifPCn53Sc3UHU30zbDijfFUiiixySUPXAjy2wBZe6E9uzSAWysLIA3ogBUm7lmHoT/6SjFGlOHaA4nglUNrbXCvNA7oLyQH634jhUBXsDtHfsAJunePMZSXYgQocmTfgIZmeVRPtqBjNAjGYNJlNy0hCtVTYmvpRmpBWrvWmvA2UBJOnGAFKCren0udrF0OBLxSD8e/TsFL7S8kU0Io/SiGEb6xK9gc6ZnPFERm0eos7m/YBbl9W5igFQ3YuIxKljoWLTX9L4ax2qEdDF912AFeBfMI9z5cArycII5AZGf+4NuRwFqB4CjcY1bszXDRUXE2fWBRjoQKdUTSc9n7rO3MDCCSMGMAWaGtsqVKTBSorrkKqIJWiL4g1Qzb7cLNUP5o47MimBukDtjzlygDWBwvnNDOnjjYQ8UYsJ/QAIbiC+wwfXn3SDEPi1VY2ZH+kz6wxY6zBSk8rsS0As0z/rolYIZhmwbQBtuZqq1BYOFhgpqazAORRZs7qyIvEJxb2O36TeQkp+pw+e14Woe0BCLJLF9J0gRHmUE8myYrIQ1LrQTj7/V5pFYqvClwE1qz9ZTUjQSnVSvPqFPQSSN4M5C8spImKVaF2Z/le5gljHP6K5SHyI8gCiwhqujunNGN+gRlnHPyrAHfV6+hsf2PXMgCs7ikbtH+uufwHAhhXPG0Wol1QTDbCWlaqh7JjkSKoypytdEEaIIV+VVACM2C3sC2TG7lwp1BDgvWTrEL1wlVAXMDuUGL5wPWLMNU0KEk94qVB9qHaGxQieEXbX/VVyiwO3gXJA+wckucFnNS8Or7VXzmiUb/CFOtAAPWfyddqBGQhFPfHUF03n72J6lg3ng5+2fHCdp0Y49rJ0wC6VWkF1AjqgSm92ADiR6vZ+x3nsLH3AIiDDCFfIDKoAqxxXE3yhVQiAoKagFE3LgMMZRTkZweHLLAiOmNUGemWHWCO2jXAISC11tI/IjXd9yTArQApaDSWn1aUVehRpXHm7rih1Rrja0oaH/hQJ1dPHz61Azbz40rXip8fq77iyAIRPhBRk00EoNxGFTShf/V0P/kJLsK4UjamUixDWakEFIR+gr3IlqNL13hX24FqnlKLluMoiTK4Uokk4yHw/RQG8hyfCfgFJfBZ38I6mBLyh37LAYEyCM1xZkKoUHxEoBE94S+1CbX4yD0YwEVAQFtnNANai7HdvvIdJaoL3wK0lfQTLinq1s0Nth6xYOjYcfq7LJJuo2V7jex0zIk5cKGI5AnquUgGkvKBliR43LjucAG+VuemrFJI2iBC823xhEEvARlEBWXDqIayBI1V2JMmd7YqWqOVwGgnWYdTjsOjni+WLT6I5AIxPFqIkGw1+f0Uq4PiK3CFHLtzHELiVb9fVWYJVfDJumYk4SSRlpCUy2l4QyKASnBz8qb6PL+bdpPyzrlEl5anXrVWruNKEwDxNrICgXGbngI2rAAeJDwsRG58giQqF/lsQB6B1ib4e2G9Y3JN9TqDxRBPLuJccl4GqlQPbOjPjcoT76mUggtf9RJzE6xxAiA3AvvPfzEYMLZRykFRaRmBgBzZtplxd59aohML11gDbx8KxO7QGYwfEGtzDl/2VBeFxmT2VKAND3CO6pgsgn3P9FwKzvJrEi/3YcjVtbV8AZiHp+v7oBOpj0qY9hb/a+2QBwNPem/4BO7OXqOSbZtaaKa0xG5laTE2MrceBhHWz+3Tfg299qLWTPmHBnDVFkQmKxSUqU/CEVL0JthAQ04fVYTL0plTIwIxB8thfZWx5bokrKjakjLVovc5Zn+/Fbk2SqFieYwoGEhcweCA6Lpk1emMNUUk3xc1UNUJ85UjI3o7it7/tYnZUJfoJmh7IO0ZoyTWimJ8/Jwg6WJqLFNNwKJHliXELGoVUTQJaSoAdygG4EgutICbvgeQmdZ3QGcooVhd7srJ5BcmnEZWK5chSEzhOIx15PhUmwXAeZrd2M5waVoBFTKPOKoeEmVAVGwNzAPc9xabF+7ddBPE6OmaSAGuxCeEQuygVwU0B7IInfR6MkE2gd11oDicBXWgbYFh3oH2AvMcb2yEQSejBC21TG2l9lowZQFAh4oMaPri3iQ/DO94K8cDb20FgJZxZhrpvqFSugpznxiRlnG9d15oNRha77LDKZQIHrTAZK+w/C/vPv2H7z8L/647af4Y28rQ/jQXoDfb/tUarseL/VWu2W4X95y3Zf3JtH6z0FUQBNFRC0zw/qHydcI4d4rJsC8TdIXBSDjI68iNqRSsoljJkRJHd4NJTGfXxTBiUJpY0JfX0wXNWxx2WJ8ffHpAdFD48Ov4WbT9RLz2qOJ4mhGJRJ+omUdFBFoPE6treEIQYkJ2RsQttztDSV8HiWlEJt/f90Jk4nuXSZr+JBZlDP3JA6stIc3aF8zepQLeLOiU0c8xYsKLtKpLK8pq96hYLVWLtSTwmgAh9JLe0jAV/u8U4MU2KTC/aiazaJh4fnZrPTw8fQVFshxkwjyAlGRjm7AsPVuziMjzd+w3vNtrdkah+lrEuFF1ExFEjofEos4zilQRzPvp2KEV0kkWBjzQdlM3xxp/HJK6vCuWkFM6o9w4+SHLm7chIz1m5I/ksZFuuDcCG8RaJRpxDXZ6JLcdMqUqR5BQsKquPhxSZcd6mhI4ScVajB8Ga93PqQigMeoSN2qArlAY8EqBo9Jk2c7dLejVKtGZYSm83FHmYBQlBbBxKJXCqRabcZFXhzLABh+wLdg5y9QFDeyQ0WMEPuhVOcNJpudrxC1rq3Mc+JxJTlJeXhETOk2pcZCKbHmnoo5UFntkjvn+SjCkXDt8jlGTlGEgPYjfiW78M8hMvUigXV9XIKD1JuoUqTA7m618UX1T4/xb+v1n+v2lU9Xp9v9ou2P87xf9nt5E/thRwI//fTPl/o9pC/992tVbw/7fE/xNDrhtrzgvc6N333CV5HBCLHs49dgE879HR88qZ47rAM6v/sa3X2JOH7OT0FPclvZgs5Il3+NqaTFwbOX1gt79Bc6sOmzgxG7rIrUm7PWDfUyRkv/kNC5yAkY2K67KKzfRy6asvzZdHXx9+w6UANNKL0HIrNf2jrHv5Z93Q2YPjpyCThNC5wGcnhw8ePz9k3BaK72kCp38KSVPDVuD87UsSWGZBrOmlbyNI28kZkui6njEFIeMMDd3eOHcGwhO2W9rYRCB4xMMpilUonvByUZKYgsQktt9GKEYI8PMNeRQ7tggVYnsxNefYusMo6uqqCpfClLKWehOAdGNy1gsI/jZbRWoCgZKAZRIU7FC2iAOFSAc3TaJU23LrmYSpEZ40WckOQso98o9kaXQgjSM/npUktgJNY7J1q5s7JW1hhO+C3GhNhjPnvBBPOY8uRpqcGBJ4v8dloSNkMu7QltkHRqu3xJQPWoriIG8dmXZssbQPuKiSNrOzasq1VibivslT00a43aW/Epe6wfo236DMZSLc5oMiaT8N7c8So3wytxLsePa91tmyt9KVUMxbzCMo5wGMKIobwhqRDIIGmkYGdpfLyyUjIzigXQs7dMZLspWEqb9mbdYbUH5T7BqJ/Y2Qm3tD3sjmUkZPmH2e9fu/SIao0P8X+v9V/r/WRP6/iP9wF65oGDqwMvzVxH/j/H+13WoV8d8K/U9B/29b/9Mw6npzv9Zu3ivI/12i/2hKPAK2PLTIRPejqoBu0P/UAeck/W+1G0j/G80i/tut6X+eWWhaOLC94XRmhej1ZbnpI1ONKt/QA9HofA5SFYiac/iIHmgjje0yFJeW7PGDV09PWQ1Glu2QJ9wOCE6Q3gNB3Ypob9i/8EopjvFCmdqqkvKkLWvR2eM0TeTPQ9ShoMRok6Err+f06NuTR4en+mzE1KEfOPAt9qUTZ5TD5NlIW93F5RGIouk8dlyxm8vDE4F0HoVDEljL7DH5PlqxtTeyFk5kEkjQvnb1nekiBJMv1EKllHUB4pbn13u8h8qqK1BpgycLZIGmyCx7OxSZSIjMFD0gV+Te1ZpvyVi7Vg5WDatH5PQGL6PlzHW8s2Sv0hpE+Au5KN4ANugcG9RTxJhj98StObXCkdJfb8vV+Y+qUvQUc6e1v9kMDtmEHWHx3hHDqAMWLNU3NHRaCTAP9SHSWP7x9R6pVE5toHSIonuEoXp8GZNTNm6U61HgOmRjCvXMRvncKbol6Q8Q27jOJbMtfC50LIv1EUpx0ryKrmk4S9nAP3moLbSV/WWhochBBFr2H44Pn1Dfor3GfjXYg6IBV94EE3IGuAizxgXfoRtIiMa5K6/MMU6zobqjQKsWOAjoHa/uN+EvlJrZzk3USuNwxcx2JmpyZgSfgGrnnicquqgDes+EA0unRp4zKhRdZlCJ0NBlAmI5s1z1af1QZN6WYUZb73IvPxwrr3fYFbywoyGG0Ii069fR7uveM0C81/3Xqtr7O62/q73WAJ1pnPk4Sh3TWEHn89VxYm8Z3OHfmT4J/XmgGmQJMePt/jyy7TWy9DmkV7QS4ZACvcJYCXtb6JNwQRHAUj7NEkCOPEzNkFfCXe2198ADvKEgYFGHPXrEHv6eNfSqTOqHE8sDWIaRprMvyY+mI2LFTFF1ys1iBIGV1QgyO5DkXP29P385H9h7xzZq1fYQZxb+3p///n/Qdjx1wxnMc3koYpDQ6AMO2GzkE4Jnk0tS/9p77b0VC8HbdL14u9a+t5CuUqkk/197Cqw8CvxwKwuArCZeaCXyyFTRFCI3W7KTUZBT9DESNPmO2i4X+r9C/5eJ/32vvV/T60a1ZdRbhQB4h+S/zP7mRzcCvkH+M2rVpoz/WufyX71VNwr57zYuIRWBABNYYYTxiJYRhmk9wBvOkTrAsoQxBhuRPCrINF7W0HH1WXL2pjl2XNs0KTBfLmoBmevmdpEJ9UpkDSsboz8IJ3O0xz3GpxB5bisgp2dLfFCVSiWRqUKMXBVK88CNaYEFWEtY2pCOe76WZZSrGwJEbK4LfQHLjNzCybgxLc3QqxtrlQ7VmXpFnInh1MfoaN2e4vkeSZrig4jBUBYxIvqbm8K9skVjRLQKXgEM6tR2g67CQ7VOrdmMWypgDDrfHZVZteuPx9RFHJlAp5HB0jGMRNYqwtKllsDSSX62dOE1bunkMG7piZ84D5Zh6TyCQrHQFPxfwf/9NfB/wv+rViv8v+4Y/4fx3+ZWbP8M5780m83V87/qRrNZ8H+3pP8XKlVSqHYwXBEZlikdxj2VlAl6jydRjMrCz7xMPubX/eu+zk54OAyKYi1CJ6S+VHrpQeKU5aCZowu4hvdj5sQy3nQE7MWTlzy0PW4/qLt7FRb7rkaqoyUGEUVmC81NySMM0soXPsVqLJ3YyExG7MRGq7ovroxys2xUr5lqectMpR73bMOArg+OvzAoLojlyVguLkYXGi6z+wUpZ8y3CAR/LF3BMl5gPxHLvGoBesC2ctIy5syPYaOJifswLjrdEfgwTjrH0XLfQM4JlskRbHMlMPhb2Oeq3tzO3W/+gr5/Vmhb2xlyyQejubN0WpS7XWRzPLOiM4ZloHcjhbtjvwU+2Ym2VMk9i7I9V2QdyLu/RX69I50uCSlrVV7Tqh22coPck1awhUt/gdsU5yzZ01mJJmTpoqPcv++czkFQsKvozzbkoRJ/i+y8gGL/gII9S2yDAgh5MEYNiEW4c3IDkmdqlKarV9d5AZGDTyI2fyqh46MM/wEttWSqJMYc35PYOj+yMeoOmCgt+7Jku6vlZmLv/bC4e6L4fNy9F6/K7MXLJNBdus/2gleB8ea6wgqYBifnS4jgLbPznqDQfU1bCTB8wOJVf8kXr+TmxmnGIRI/vEwCOeb93vj+GwUOgoZxKS4S22NA9sXG1AkOmtGhjjT5j1HF3+sD9uCYWyMfIEEVd8df52O+nDtl3vM0wNqLzJ7XSj+y/oRyePIbUDw0iUnbQeSOeooSpxRJuZMnDol09US/08Tj9MWr3rnTX4n3ulY8juYmr9GtxUDvJYxfvMRP3K84D2yZeB7ZI6SoLhllk930ij33BtdTuQXm2ZexqiaR6yZ5wALGTNDPlJ8lIkIlYX3kPT1J3FIR4VMnVfqILpu03vN4OTIwzi7AFii0TMJf3qcEMnmFJ9HKmZiPqRG2KwEzyzrwAlpis5DWqbP8riNFDzvpsJPeWT/ZriOnUVjgVSgQoylltwpjn4fsn0Tkp3r89XqufM8hQ77vk1zf6bN02c0BA5tHEJdwzrSCojN1WRVJOCJnVZwkk0TzLfPjNdLBgq5oayGMwg4viR+2gkXBHb3ZYzDsu8xIq4TZJ7uKCfcodBfuhsnmIcYmJ2VEPBoYzuadnauxcvLF1dl1EmmPw8oLeDTbBW6Y1fNxgE+SOMAYdI94qi25oV08PwaO9APDJGA6nvnk5ZYcx18nGWYRbf0vk5To8b6TTw7TDdNT5EDPlFwKDzb4QsQK5HuDfN4mndf4agL8QycbTJBHJoQCTvUR7v7KmIIUh5PirNG2LgVAjGSQQaH8wg1dHuPPA/7ir1fFVeh/Cv1PYf9f6H8mwZwbs/0ECqCb7D+NZn3V/7fWbhT6n1vS/6CXL6yf0Ty0efAbVbjtHsMaC+zDy4bWkSHykwMfcYvIn0+mAS52PKTPmb3kYVOFAAssqZSUo8AG5rZEwYFenUDqhWNlHTZRspzZMz9cmpbr+hjkeaQzMk5DjRG3nkpRlNZodmJbLupfBnjmFcrt68dNch2NMPdMFDd0qOStbW/KsLrlfBTxkhVhvdvdVpGJoQMqYHyUkjwPK0mM5wFwZ1STGofhf0gmUwBOFIF5QXwWZEhOfTFN9GyEATHN69KCx71esyvcYt66ajnV6zRBfOVHXlKs89Qfk0oW3pX8dJq1+Om4BQpM+UkPGDURIxpZUOTU+KNGkYUQYTC+qDz/iEdJfz/eqHRszs5OvYoRokuZ1GiVFnOuU2Si4M4oJdvZQPzKhx4og/uu5H+MJYiG9zqNfq7WaAkwDH20xVM3CegyIx3/0Wpgki2ZSwAu3iKTt4hghvraCQ+RnTDHGTiigLAqcwoemSCxgGTmZIDqXgL2dQm94u1cK+xZEC/NoTWc8i58AEhLP0RZUloDCmRY1Z0QtlBBqLoyOcIIjMj3D7tHRcghqf6wEfEwMjaefIlgpGI0LT1sQMD1/cOUuo+LMeJBzHmIZjlKa+MCY5UZOSH5kDv1htzQypXktfcMKpd7EuEmUk8y0ol2kDuwIhFs1o+sSKWjEyHvKJvJs/K3IwIV+/+F/FfIf1z+q1YbLf1eu9VuVIv9/7sk/yHxJ78eqTf7iHLgTfafrZYh7T8b9WYb5n+zWqsW8t8tyX9fwajjCSC5Tc8Oj7wqfEn4JwrD6kRsOnadIMDdipF/4dF51uR+16z+GpjY0JlM+fHJl1W9zQO0Bhb5lYAkaJVCeOHPKp7vROmpoUPLW1ggeYII6i9AooiGtmdTsEoMxmQRx8MdV6gpWiIcSn8w/p4wmIuHa7Igin43xmwVZ3AeJQeTH3pTjOJUGmx0BjxgoTfhhwfxbuliM9aE9ySSvdh4essgdWTjDU/Pb0nO/FvxH8xkIfe7jT6E+a1EuZMo+qA/TIZGlb3UZw4en6hmdhx5XVfnvc+Jf/28f62sbjNqeHoWlanCEIsjy2Z6PAVh3AMBVlUNowZcuFHDHg3SLc002CzAR0drjwkIpCh715rNMlPpVFb6U08OIQXYziHlvtixGEx0wibuI4Y5gHWGyi+cUTwFDhpkP14Qfz21sc/8PTYG94TSDVSQXwIXhD1VGsHsrbo57ilJtZG1sCV0gmuFj1VyRtbOzjmeZSRNZ661Usor047A1rFPkXadaT5gqTMTejZ+dC76r4P/q63zf9WC/7sV/q+V8//ZbzXq+r1WrdUuuL87xf8FQAmt0DZpdfvIewA38H+thpH4/xjNVgPjPxjtIv7DbfF/D9GJg1mMTtfmzrgV1HQ6dj4QBDFMGT9kstNEp3Pk8RJX5NQTWdNLFfYoPbIxYn+oVFCPRVX8IXH3jfgpVRgRFJ2tVVi6x0FEQSEyDBdXP0OJRyt8KnGlKpnnDTBM6a7R/DUwnBh2YMyZWNE6ZEB5XjRrQlu8EZoMirPe1DEwWjE2Ef1e4P0fhKYPln9ZUBrQIrTZN0cvGXpsQ+eQ+UXmVFobRfyRDBZE8WjkgoEy+LF/9oiAc0pcbtITzDOewxjkm4Ypn7zsJMaxeNqXPaRTuDBH5lDMkXClvpja4jQE0WGCDtnZeBVS5rK9TKKkW8ALLaPNxq/YLc5H892UDzwC4UcYotJwZ80aM0E13ufVlU+f5dI3GqYmiLjRL6q+3Y4UgzxszlPdnEng08Y8wCHf5DyWS19rQGc2mHiuBLywdOr/WtgLwJ6NQS96Hb6XgsWsSR5k0bI1dkledZwmT42EV+QUH/jk6Dw1gcR9ngS11bFroR2fPx7Lk5SRd45E8uTgN2E6Nc6bTlFmtst6X0JRuRMO8WAFbqyWnq6AhTqrxyRkUkmjtE6Wz1ehZVAD2l0Jtzb+QnVA3jA0+ZoCxGZPqhM7WFASCWqL9VPcEfYa2iwmmCnaTyEnyJTtPOotnM7CIfO7JFn/gC1wC5BCjnC8v4I0e3vZRJ1qbcSDjoioINiyA2p8apRGRoY2CZFUaQq/N8GWECQJpq0HIrHPs6FILO+mEjLxLPJFBN5EyZjUITEjhEiO7M1Ir4HYqqEgJZbXP2C5bU40rESB0nMAP2nRoN0h3H65qmYOKT7PRzN5E2CqA/ZVmX2HAmQayCTJwVPzEB5rs4B3xSStBkGkzM7Tys7sAGV+jg2WLugFx4o3AQLPE3ujmDITw/g8PaFZCp5jZXuVCrzBw//s8zWvjg5HocwpgoAZyTmCAsW9HIpf96/zFpq+QwGaAN5580Xcd78URyzTopPAHa0uIZe2au2IXb2M0Oi8Ud10ZDJelzBtLo0yW8Lv0oDSL1GZ4eG+Od7hKSTwIXm3FO9AoL4ss2ApTi3RjSbbYeqlAV27rGokeidvl/h2WV052vRDBhqW3Svo2LUc6x7WjQ2tVoKl1sE2fYXN3oUnOpkIP17Cx0v+8Tvs2y48rRx4ygkcoP7A911VwNFJ4KjjpqXG7gN/VuXTgMgMYk3+2HD/AzBnpQ855EmdA7I7zpd8t/krgN130K2GVt545DgeaHoT9kUJ7tnX8jBZMkffuFKsYmLS6zwijtcmtdNfw71xJkjRr7JBijpJfhGkaLw5RJIo5yJzjs/GMEyrK+veFQHhWvmA8Exi6dlWfRJ4aZy+pzkMc1BO39JaOKWMvuq8nGElsvidU1du0FalZUTbyxA0afMuMdd3QaZkTbymlQ/Fgat8TB8qdvfzxDLlcw3S8ocDntY/hzd5xXaZf4nwS5Tlv+9mBKAi/msR/zWn/zOaGP91v20UCsC7pP8jXcVPc/znzfF/2tVU/1fH+V9rtNuF/u82rm3xf7jt7J1xaeZM8Ic6NGe1TeIsyzRCD7fSI/cgtFxUyIQP0myJzXO2Uc/T3OLKG232u131lqQYjGR4K/0VMyIyZKPzZVY2dWXAHvw9E6qHhBkLe58TY/V5//p1jE/EqH/e7+jG+LqCL4BbF48RU8nQG99yfzn5QXsd8yOHqAS8wy/18bWS8n3CE/+K/LuyFpKVWOvo1fE1m93ZYI2/WP6vsP/72fi/vP2f0Wgbes3Yb7WqRfzHu8T/hXMPXShM3PODRWP6sef/e85/Meq1luT/2ka7Tfu/rcL/61auT3+1N3C8vYEVTUufsi8p5jLt+86sOHQuUesiDfx5yHV2gK4RMytcMtfxaPM2SYFINMYidNeflOxL4uBenjz45vTLo5Pnhyen5qvDk4dHp09f/r5ro91ZyV6o2hWzgXtiSrfLPjPYZzX2WV05YMEynqLn2Hp8KiY4O7477IwuTciWRKVhq9t/idrpsxrvwGd1RnuGSbM/uzKu4etnDf69dv83BnvLJqEdsMohU/6Oe1i85fzcW2TmNGjgdanHKuO0ETlfoT10FefF9dnbt6u9yUdbZTKYy3rbxXve3s1VMRGukn2o1xKj7UWGMJMuORgDJddvfpAjspGocmX5CumpVuH2AhtDYh6wkQ8MpL1gn3ksY+V2kHvFMpF5cA+qqTCTCoWsPUzWXam4j4eDXuXKkCWSKg8HZYQ7bgVdL/i/gv/7Efxfs9He1/fbjdq9wgLwzvF/5Of7kXm/D+H/Gq1WGv+xgfxfvW60Cv7v9vk/OgqqEtkx5wHV5FAmDd3tge9D1/QOi84cYCDIrgdDAXCTNjSJ4RwVQ31fpH8IA3jAHnY3+5tnuKtnJjIZP5Kl+uzhBiaKl5hwTu+L7p3hkjD2dq5ZnDv5uA3jZf4Ypi7LzBHr9gJZtwz3lbBMxJxRRyQLzAFifvZiez+2cuAJMCUHDj1LeG5RoAgEKYwh8+x3vu71lgmX9x/ZtgSe721dvj35GjlLSTLKg2fPHh99c/gLWhkL/q/g/zL8X6PRbOlGq9msNwr+7y7xfxSwYhjMdVr2Pv78f5//R61Wk/xfs4rvjXrTKPZ/b+UaOeMxrH8TJ2YWIcEe8R17I3sIvyEuqIPN70t8ma3ds6xR3db1ljEcjQY1ZtBglyqVyrYSS7u7u1tL/eILVqlXq+U22+U/8GLoWlHEXoaWFwFfM7PDxzy96nn6c380d+2ViJkUMtOOfHdO4acyxtzcvDsGFsnO2XivWyqObSs20dCPQsomZe3tiezrOYa+H44ic1qWdxdkte2OdROjNfGXaqmyySIyqa2cveUBnro8yFBp9wdnVHlOBIdp8h1101RFVChtU9QpfqShApRA0dZ7uOEV9W/ozwJeAvQyHPFQQdB5dR0kH1BEBO2zRxL26qYeaqVtiBv4kYMDZYIo4WOksjUU3pRCIPO4vj9s1Qa6vt9u2Xbd3orMG8tYQ+uNqRDBmzXEb/qboPexSHwoQ6ydOp69HcPxUkEMQLjNY9vMo2i9Vmbv+bhqmZu3DuYGtwh9x8uWgq+izjr+xrYX+WGUxCf7ox36kaoaGIcJfSXWUVlbx+UfUshfhtbbMBB6fQFSkipasiHZp2w8D+OpHcKQSVoyAuFvOMWjDKQ/FuG++gYWFvQLG9gsssb2VrTH1D3sXV8Si8wrnepRNZ3Xom5H+zTglBmE/tCOoAtrmL8lkUT+/Ub93rip67XhcDRujrci/7Zi1vB/W0KcAkYDpwD9hUd+MCmO38LBAHV6LIl9Ei9sUSuJmXIKxR7L8sS0UBQF/5cYR090KgLU8BxEDIQqj9XtljOkvAv9219BTKRT3hiouze0zeQULDpyoMNx9oeX/Jdh63tblMclrjHois2b/LcMBcguZyupeItx34VutmLbwgU6PxsAgVpHsvw3gVvDants3zN0HYSM0aB1bytureReQ6mV74RJNaKm/Cchpxg77tWzh9bwbOBvo6OpUxbQN0AdJgdYkAITo49kFmw+3EMriPkxZo4H+UwK0NblAeEtoNz40XIpr3i7ZRn/C4v7GIv7BmhgyM4FuWGZw7MgNi9CdBUJVc7IZCFjer4J8KVkmponcLJX3aR7fKzqbRor+vlBY5WWkzJ5q/BKP20EXDqQ+TmfjsjHmbHroOUYxo+5YJzFrbWIxaWfBBCviPgdee7yFwiEQSS82GQvtE0AKrQOhf6v0P/9XPo/o2UYer11r9XaL/R/d1H/h4z64COfAniT/0e7Xk/s/xpGHe3/ao3i/L9buUBmOsYRFzHcSVhDmdYexujS7bFHx992GAavIwcDkMSBF6zwbTzxgngP0hlQsD5AoBlgVDaMCA/DTp4lFDeER2DfHDGE3iI2cnHGpO1TjJuSPSw6I1gKS7V8Ln2T5Jm4lGTFx/XQ0xg2b3MtasIxIYdzwOh4r2xh6izD62MaZJ4U/kZB/UAutS7FSLhBdSbcDH1zYZEXTcSZqZXipZuEIkJgl7cHwC6zsTuPpsKRZUjuM4lf8SPgwOahvR7rXfrS8lfVKg/2foCMp46xE7CERw+OTTxJyzw+OjWfnx4+wsO0qhj8xD8rcydpSM0DjJSyB5ql0f+wmOEifuS70Ksxd3h+dPTs6MR8+OSkdvLkocZ9WCiKQ0/h+9YKRbcOhduw5cHIuEqfM5jrAcTxzTG2mY+gCg1BRxljNSGPt8hTklDD8VflP90A8BZjB3UjEXiBD0BADvGi7OxAGCn8FV7E9mEy5FlFCo/1AE3pKeQWE9EpTT2eK8LI3ukBYLlUeMKWE1EEl2a/nx/zvyH+r/D//dn4v8L/905fVhD8RF6/H8z/1evJ/q/0/602akX8v9vi/56E1sjxgXGZ+R1pWMXRYsXTAm95gMitIeKIvZvwAi00RSv93J69SdiyD0gbTa0QXXf5qUVdhTYizRjSbztad+M5uBTSjFhk04pVakAZmQJAIaEP28KO5Q57TQ8v5cHAbubCYraDevXNnNiBuJfRXjIxz/yz5ISdJF4ZfILmdkS8m2FseROXQt7E8wBu4BtGyNGyz7UOPifBnKt0gEk2+NnNXB9BLpx7yK3J01WlLzKeXQrMZBJURxQqoshtOqMVkq/5OPMykYU7ExuBEwv3THpqMmDpCahlPNtUHvlJD9BPYs3GylWUukSzLxg+5p2dmYrvEi/n2vhaU7T8Oa682aF/QTGdeisVi7NHy8kBqyvNEdyhYAbxTV3rb6yAQwo6WqbKSnTM9CTUH7r+8CwCtjR2QWL59//y7vt/YEaN/fnv/5m9/Lf/NWNnzrs//d8Ze/enf/Gme//+T+++/yeHxf/vf7/7/l/wALDQh7+8wRpOdiIgImKd/twKzzA6vKp8+ul6YWyPvac49uf/9F/FnTeZv/v+nz08N3f67vv/5kA25933/zh77b2EghxRmjr13/3p/wzFEwUEXSleY+6//U9PF9sesv8n/oWaDRJIsgq8J3FFJa98JXDwHHLXGthuV3n3p/8OFQBBWLLFuz/9q4dkIckuSwX8ns88dWXDns5TptJPkDqq7wkUUMbQqnM7iS0gKn8EEPjPbPbu+38dYr1nvLRT10FLGIOfaNxEccUOukaS6yUed4/pB7HHczycx7EPA4OjomTnAHx7YrmuHS5VkflrGDKMnASApa1R7FhEwQlia8BzPIalgXqiTkmEwBAIMq4mR2E10hQezkvc0ZlE/FagroixHnv60HWGZyqQgDKeKZYSAZzlhMBQcR/pxMzXXWsOIqtKVLtr6fRbRAYo9P+F/Pdj9f/Ne0bN0Kv37tXu1WvFTLoDl+CK6eBPDA38M8h/IAAm+v9as2Zg/Kdms9D/38r1KYtteTYPMfykm2c13Wjoxu4wmMOKm9rdRqyp16o6yHhjy4miCnxnhm40dYMHexwuKlyArCAz4OIhQE29Cv/u1ROxsKXXMd4w1XO/CxWVshXc7zb0RrOUFH+/a+j7pc1lY9r9EgUhLwGb5voXJV4HfqgW5Ku4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu4iqu47tb1/wEOYWfFABgBAA=='
tarfile.open(fileobj=io.BytesIO(base64.b64decode(CODE)), mode='r:gz').extractall('.')
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip -q install faiss-cpu gradio
!git clone -q https://github.com/facebookresearch/sam3.git /kaggle/working/sam3 && pip -q install -e /kaggle/working/sam3
import torch; print(torch.__version__, torch.cuda.get_device_name(0))

In [ ]:
!wget -q https://data.vision.ee.ethz.ch/csergi/share/davis/DAVIS-2017-trainval-480p.zip -O /kaggle/working/davis.zip && unzip -q /kaggle/working/davis.zip -d data && rm /kaggle/working/davis.zip
!python scripts/prepare_davis.py && python scripts/make_hard_queries.py && python scripts/add_distractors.py

In [ ]:
# 1) Đo VRAM & tốc độ thật (DINOv3 + SAM 3 trên keyframe)
!python scripts/gpu_bench.py

In [ ]:
# 2) Index lớn (70 video): CLIP baseline frame, DINOv3 frame+grid region
B='data/davis_bench_large'
!python scripts/build_index.py --videos {B}/videos --out data/L_clip --model openai/clip-vit-base-patch32 --regions none
!python scripts/build_index.py --videos {B}/videos --out data/L_dinov3 --model facebook/dinov3-vits16-pretrain-lvd1689m --regions grid --dedup 4

In [ ]:
# 3) Index region bằng SAM 3 trên keyframe (dedup aHash)
!python scripts/build_index.py --videos {B}/videos --out data/L_dinov3_sam3 --model facebook/dinov3-vits16-pretrain-lvd1689m --regions sam3 --dedup 4

In [ ]:
# 4) Đánh giá: baseline, DINOv3, SAM3-region, SAM3 re-rank top-20
for Q in ['object_hard','object']:
    !python scripts/evaluate.py --index data/L_clip --queries {B}/queries_{Q}.json --modes frame --out results/K_clip_{Q}.json
    !python scripts/evaluate.py --index data/L_dinov3 --queries {B}/queries_{Q}.json --out results/K_dinov3_{Q}.json
    !python scripts/evaluate.py --index data/L_dinov3_sam3 --queries {B}/queries_{Q}.json --out results/K_dinov3sam3_{Q}.json
    !python scripts/evaluate.py --index data/L_dinov3 --queries {B}/queries_{Q}.json --modes both --rerank sam3 --videos {B}/videos --out results/K_dinov3_rerank_{Q}.json

In [ ]:
# 5) (tuỳ chọn) SAM 3.1 video predictor – đo VRAM đỉnh trên 1 video (API theo commit release SAM 3.1; chưa test)
import torch, time
torch.cuda.reset_peak_memory_stats()
try:
    from sam3 import build_sam3_predictor
    t=time.time(); pred = build_sam3_predictor(version='sam3.1'); print('load_s', round(time.time()-t,1), 'peak_GB', round(torch.cuda.max_memory_allocated()/2**30,2))
except Exception as e: print('SAM3.1 failed:', type(e).__name__, e)

In [ ]:
!mkdir -p /kaggle/working/results && cp -r results/* /kaggle/working/results/ ; ls /kaggle/working/results; cat results/gpu_bench.json